# 🎯 Stack 2.9 — 128K Context Fine-tuning

Fine-tune **Qwen2.5-Coder-1.5B** with **packed 128K context windows**.
1500 examples → packed into long 128K sequences for massive training signal.

**Runtime:** Runtime → Change runtime type → **GPU (T4 16GB)**
**Time:** ~6-8 hours on free Colab T4

## Step 1: Clone Stack 2.9 & Install Dependencies

In [ ]:
# Clone and install in ONE cell — use ! for shell commands
!git clone https://github.com/my-ai-stack/stack-2.9.git
!pip install -q transformers peft datasets bitsandbytes>=0.46.1 accelerate huggingface_hub scipy
!pip install -q torch --upgrade
import os
os.chdir('/content/stack-2.9')  # Change to repo dir for subsequent cells
print('✅ Cloned & installed. Working dir:', os.getcwd())

## Step 2: Login to HuggingFace

In [ ]:
from huggingface_hub import login
# 👇 Put YOUR HF token here
login(token="YOUR_HF_TOKEN_HERE")
print('✅ Logged into HuggingFace')

## Step 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/stack-2.9-128k-output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'📁 Output: {OUTPUT_DIR}')

## Step 4: Download Training Data

In [ ]:
import huggingface_hub, shutil, json

print('Downloading from HuggingFace...')
path = huggingface_hub.hf_hub_download(
    repo_id='walidsobhie/stack-2-9-tool-examples',
    filename='tool_examples_combined.jsonl',
    repo_type='dataset',
    local_dir='/content/',
)
shutil.move(path, '/content/tool_examples.jsonl')

with open('/content/tool_examples.jsonl') as f:
    lines = f.readlines()
print(f'✅ {len(lines)} examples ready')

## Step 5: Train! 🎯

In [ ]:
import subprocess

result = subprocess.run([
    "python3", "training/train_extended_context.py",
    "--model-path", "Qwen/Qwen2.5-Coder-1.5B",
    "--data-path", "/content/tool_examples.jsonl",
    "--output-dir", OUTPUT_DIR,
    "--context-length", "131072",
    "--lora-rank", "32",
    "--epochs", "3",
    "--batch-size", "1",
    "--grad-accum", "16",
    "--lr", "2e-4",
    "--use-packing",
    "--push-to-hub",
    "--hub-model-id", "walidsobhie/stack-2.9-128k-context"
], cwd="/content/stack-2.9")

print('STDOUT:\n', result.stdout[-2000:] if result.stdout else '')
print('STDERR:\n', result.stderr[-2000:] if result.stderr else '')